# Workshop 4: RAG with Hybrid Search

In this workshop, we'll build a **Retrieval-Augmented Generation (RAG)** system that:

1. **Embeds** documents using local MiniLM-L6-v2 (no API calls)
2. **Stores** embeddings in SQLite with FTS5 for keyword search
3. **Retrieves** context using hybrid BM25 + semantic search
4. **Generates** posts grounded in your knowledge base


---
## Part 1: Setup and Database Initialization

We'll use SQLite with FTS5 (Full-Text Search 5) for efficient keyword search.

In [1]:
%%capture
# Install required packages (run once)
!pip install -q sqlite-vec fastembed numpy openai mastodon.py

In [2]:
import json
import os
import sqlite3
from datetime import datetime
from pathlib import Path

# We rely on Colab Secrets or manual input, so we don't need dotenv here.
# from dotenv import load_dotenv
# load_dotenv(Path("../.env"))

print("Standard libraries loaded!")

Standard libraries loaded!


In [3]:
import sqlite_vec

# Initialize SQLite database with FTS5 and sqlite-vec support
DATABASE_PATH = Path("tutorial_rag.db")

def init_database(db_path: Path) -> sqlite3.Connection:
    """Create database with embeddings table, FTS5 for BM25, and vec0 for vectors."""
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row

    # Load sqlite-vec extension
    conn.enable_load_extension(True)
    sqlite_vec.load(conn)
    conn.enable_load_extension(False)

    cursor = conn.cursor()

    # Metadata table (stores content and metadata, linked to vectors by rowid)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS embeddings_meta (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            source_type TEXT NOT NULL,
            source_id TEXT,
            content TEXT NOT NULL,
            metadata TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

    # Vector table using sqlite-vec (384 dimensions for MiniLM-L6-v2)
    cursor.execute("""
        CREATE VIRTUAL TABLE IF NOT EXISTS vec_embeddings USING vec0(
            embedding float[384] distance_metric=cosine
        )
    """)

    # FTS5 virtual table for BM25 keyword search
    cursor.execute("""
        CREATE VIRTUAL TABLE IF NOT EXISTS embeddings_fts USING fts5(
            content,
            source_type,
            source_id,
            content='embeddings_meta',
            content_rowid='id'
        )
    """)

    # Triggers to keep FTS5 in sync with embeddings_meta table
    cursor.execute("""
        CREATE TRIGGER IF NOT EXISTS embeddings_ai AFTER INSERT ON embeddings_meta BEGIN
            INSERT INTO embeddings_fts(rowid, content, source_type, source_id)
            VALUES (new.id, new.content, new.source_type, new.source_id);
        END
    """)

    cursor.execute("""
        CREATE TRIGGER IF NOT EXISTS embeddings_ad AFTER DELETE ON embeddings_meta BEGIN
            INSERT INTO embeddings_fts(embeddings_fts, rowid, content, source_type, source_id)
            VALUES ('delete', old.id, old.content, old.source_type, old.source_id);
        END
    """)

    conn.commit()
    return conn

# Initialize the database
db = init_database(DATABASE_PATH)
print(f"Database initialized at: {DATABASE_PATH}")
print("sqlite-vec extension loaded successfully!")

Database initialized at: tutorial_rag.db
sqlite-vec extension loaded successfully!


<cell_type>markdown</cell_type>### Understanding the Database Schema

> Add blockquote



We use **two storage mechanisms** for different search types:

1. **sqlite-vec (`vec0`)** - Native vector storage for cosine similarity search
   - Stores 384-dimensional embeddings compactly
   - MATCH queries use optimized ANN (Approximate Nearest Neighbor)
   - Returns cosine **distance** (0 = identical, 2 = opposite)

2. **FTS5** - Full-text search for BM25 keyword matching
   - Indexes words for fast exact-term lookup
   - BM25 scoring ranks by term frequency/document frequency
   - Returns negative scores (more negative = better match)

### Part 2: Document Chunking

Large documents should be split into smaller chunks for better retrieval. We chunk by `##` headers to maintain semantic coherence. But it can be different for every knowledge base

In [4]:
import re

def chunk_document(content: str, filename: str) -> list[dict]:
    """
    Chunk a markdown document by ## headers.

    Each chunk includes:
    - The document title (# header) for context
    - The section content
    - Metadata about the source
    """
    # Extract document title
    title_match = re.search(r'^#\s+(.+)$', content, re.MULTILINE)
    doc_title = title_match.group(1) if title_match else filename

    # Split on ## headers
    sections = re.split(r'(?=^##\s+)', content, flags=re.MULTILINE)

    chunks = []
    for section in sections:
        section = section.strip()
        if not section:
            continue

        # Extract section title
        section_title_match = re.search(r'^##\s+(.+)$', section, re.MULTILINE)
        section_title = section_title_match.group(1) if section_title_match else "Introduction"

        # Build chunk with context
        chunk_content = f"[From: {filename}]\n# {doc_title}\n\n{section}"

        chunks.append({
            "content": chunk_content,
            "metadata": {
                "source_file": filename,
                "section_title": section_title,
            }
        })

    return chunks if chunks else [{"content": content, "metadata": {"source_file": filename}}]

# Test chunking
sample_doc = """
# Our Company

We are an AI consulting firm.

## Services

We offer enterprise AI strategy consulting.

## Team

Our team has 20 years of combined experience.
"""

chunks = chunk_document(sample_doc, "company.md")
print(f"Document split into {len(chunks)} chunks:\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk["content"][:200])
    print()

Document split into 3 chunks:

--- Chunk 1 ---
[From: company.md]
# Our Company

# Our Company

We are an AI consulting firm.

--- Chunk 2 ---
[From: company.md]
# Our Company

## Services

We offer enterprise AI strategy consulting.

--- Chunk 3 ---
[From: company.md]
# Our Company

## Team

Our team has 20 years of combined experience.



---
## Part 3: Local Embeddings with MiniLM-L6-v2

We use `fastembed` to run the MiniLM-L6-v2 model locally via ONNX. This means:
- **No API calls** - Everything runs on your machine
- **Fast inference** - ONNX runtime optimizations
- **384 dimensions** - Compact but effective embeddings

In [5]:
import os
from fastembed import TextEmbedding

# Suppress Hugging Face token warning
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

# Initialize the embedding model (downloads on first use)
print("Loading MiniLM-L6-v2 embedding model (ONNX)...")
embedding_model = TextEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Model loaded successfully!")

Loading MiniLM-L6-v2 embedding model (ONNX)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

model.onnx:   0%|          | 0.00/90.4M [00:00<?, ?B/s]

Model loaded successfully!


In [6]:
def generate_embedding(text: str) -> list[float]:
    """Generate a 384-dimensional embedding for the given text."""
    embeddings = list(embedding_model.embed([text]))
    return embeddings[0].tolist()

def generate_embeddings_batch(texts: list[str]) -> list[list[float]]:
    """Generate embeddings for multiple texts in a batch (more efficient)."""
    if not texts:
        return []
    embeddings = list(embedding_model.embed(texts))
    return [emb.tolist() for emb in embeddings]

# Test it out
test_text = "Artificial intelligence is transforming how businesses operate."
test_embedding = generate_embedding(test_text)

print(f"Input text: '{test_text}'")
print(f"Embedding dimensions: {len(test_embedding)}")
print(f"First 10 values: {test_embedding[:10]}")

Input text: 'Artificial intelligence is transforming how businesses operate.'
Embedding dimensions: 384
First 10 values: [0.035499076629084546, 0.00839253664254505, 0.05062492341224551, -0.007122231672038325, -0.01755316620916984, 0.006484544607462985, -0.001429692512319824, 0.0006524717970323388, 0.014390853328519674, -0.02782975465926638]


In [7]:
import struct

def serialize_embedding(embedding: list[float]) -> bytes:
    """Serialize embedding to binary format for sqlite-vec."""
    return struct.pack(f'{len(embedding)}f', *embedding)

def save_embedding(conn, source_type: str, content: str, embedding: list[float],
                   source_id: str = None, metadata: dict = None) -> int:
    """
    Save an embedding to the database.

    Inserts into:
    1. embeddings_meta - content and metadata (FTS5 updated via trigger)
    2. vec_embeddings - vector for similarity search (matched by rowid)
    """
    cursor = conn.cursor()

    # Insert metadata (FTS5 index updated automatically via trigger)
    cursor.execute(
        """
        INSERT INTO embeddings_meta (source_type, source_id, content, metadata, created_at)
        VALUES (?, ?, ?, ?, ?)
        """,
        (
            source_type,
            source_id,
            content,
            json.dumps(metadata) if metadata else None,
            datetime.now().isoformat(),
        ),
    )
    rowid = cursor.lastrowid

    # Insert vector with matching rowid
    cursor.execute(
        """
        INSERT INTO vec_embeddings (rowid, embedding)
        VALUES (?, ?)
        """,
        (rowid, serialize_embedding(embedding)),
    )

    conn.commit()
    return rowid

# Embed and save the sample chunks
for chunk in chunks:
    embedding = generate_embedding(chunk["content"])
    save_embedding(
        db,
        source_type="business_doc",
        content=chunk["content"],
        embedding=embedding,
        source_id="company.md",
        metadata=chunk["metadata"],
    )

print(f"Saved {len(chunks)} embeddings to database (using sqlite-vec)")

Saved 3 embeddings to database (using sqlite-vec)


In [8]:
# (Optional) Environment inspection - no longer needed as we use manual input
pass

### Embed Business Documents

Let's embed the actual business documents from the project. Add your files to the business-docs folder.

In [9]:
BUSINESS_DOCS_DIR = Path("business-docs")

# Create the directory so it's visible in the file browser
BUSINESS_DOCS_DIR.mkdir(exist_ok=True)

def embed_business_docs(conn, docs_dir: Path):
    """Embed all markdown files in the business docs directory."""
    doc_files = list(docs_dir.glob("*.md"))
    print(f"Found {len(doc_files)} document(s) to embed")

    total_chunks = 0
    for doc_path in doc_files:
        print(f"\nProcessing: {doc_path.name}")
        content = doc_path.read_text()
        chunks = chunk_document(content, doc_path.name)

        # Batch generate embeddings
        texts = [c["content"] for c in chunks]
        embeddings = generate_embeddings_batch(texts)

        # Save each chunk (to both embeddings_meta and vec_embeddings)
        for chunk, embedding in zip(chunks, embeddings):
            save_embedding(
                conn,
                source_type="business_doc",
                content=chunk["content"],
                embedding=embedding,
                source_id=doc_path.name,
                metadata=chunk["metadata"],
            )

        print(f"  Saved {len(chunks)} chunk(s)")
        total_chunks += len(chunks)

    return total_chunks

if list(BUSINESS_DOCS_DIR.glob("*.md")):
    total = embed_business_docs(db, BUSINESS_DOCS_DIR)
    print(f"\nTotal embeddings created: {total}")
else:
    print(f"No markdown files found in {BUSINESS_DOCS_DIR.absolute()}")
    print("Please upload some .md files to the 'business-docs' folder!")

No markdown files found in /content/business-docs
Please upload some .md files to the 'business-docs' folder!


## Part 4: Hybrid Search (BM25 + Semantic)

Hybrid search combines:
1. **BM25 keyword search** via SQLite FTS5
2. **Semantic search** via sqlite-vec native cosine distance

### Why Hybrid?

| Query Type | BM25 | Semantic | Best Choice |
|------------|------|----------|-------------|
| "Emanon AI" | ✅ Exact match | ❌ May miss | BM25 |
| "help with machine learning" | ❌ No exact terms | ✅ Semantic match | Semantic |
| "Emanon consulting services" | ✅ "Emanon" match | ✅ "consulting" related | **Hybrid!** |

### sqlite-vec Benefits

| Aspect | JSON Blobs (old) | sqlite-vec (new) |
|--------|------------------|------------------|
| Search | Load all → NumPy | Native SQL MATCH |
| Speed | O(n) full scan | Optimized ANN |
| Storage | ~4x larger (JSON) | Compact binary |
| Scalability | <10k vectors | Millions |

In [10]:
def bm25_search(conn, query: str, limit: int = 100) -> dict[int, float]:
    """
    Search using BM25 ranking via FTS5.

    Returns dict mapping embedding_id to raw BM25 score.
    Note: FTS5 BM25 scores are NEGATIVE (more negative = better match).
    """
    cursor = conn.cursor()

    # Escape special FTS5 characters
    safe_query = query.replace('"', '""')

    try:
        cursor.execute("""
            SELECT rowid, bm25(embeddings_fts) as score
            FROM embeddings_fts
            WHERE embeddings_fts MATCH ?
            LIMIT ?
        """, (safe_query, limit))

        return {row[0]: row[1] for row in cursor.fetchall()}
    except sqlite3.OperationalError:
        # No matches or invalid query
        return {}

# Test BM25 search
test_query = "" #FILL THIS OUT WITH TEST STRING
bm25_results = bm25_search(db, test_query)
for emb_id, score in list(bm25_results.items())[:3]:
    print(f"  ID {emb_id}: score = {score:.4f}")

In [11]:
def semantic_search(conn, query_embedding: list[float], limit: int = 100) -> dict[int, float]:
    """
    Search using sqlite-vec's native cosine distance.

    Returns dict mapping rowid to cosine distance.
    Note: cosine distance is in [0, 2] where 0 = identical, 2 = opposite.
    """
    cursor = conn.cursor()

    # sqlite-vec requires 'k = ?' in the WHERE clause when using a parameterized limit
    cursor.execute("""
        SELECT rowid, distance
        FROM vec_embeddings
        WHERE embedding MATCH ?
          AND k = ?
        ORDER BY distance
    """, (serialize_embedding(query_embedding), limit))

    return {row[0]: row[1] for row in cursor.fetchall()}

# Test semantic search
test_query = "REPLACE WITH A QUERY YOU WANT TO TEST" #TODO: REPLACE WITH QUERY YOU WANT TO TEST THAT IS IN YOUR DOCS

test_emb = generate_embedding(test_query)
semantic_results = semantic_search(db, test_emb, limit=5)

print(f"Semantic search for '{test_query}' found {len(semantic_results)} results:")
for rowid, distance in list(semantic_results.items())[:3]:
    print(f"  ID {rowid}: distance = {distance:.4f} (similarity = {1 - distance/2:.4f})")

Semantic search for 'REPLACE WITH A QUERY YOU WANT TO TEST' found 3 results:
  ID 3: distance = 0.8458 (similarity = 0.5771)
  ID 1: distance = 0.8707 (similarity = 0.5647)
  ID 2: distance = 0.9364 (similarity = 0.5318)


In [12]:
def normalize_bm25_scores(bm25_scores: dict[int, float]) -> dict[int, float]:
    """
    Normalize BM25 scores to [0, 1] range.

    FTS5 BM25 scores are negative (more negative = better).
    We invert so that best match gets 1.0, worst gets 0.0.
    """
    if not bm25_scores:
        return {}

    scores = list(bm25_scores.values())
    min_score = min(scores)  # Most negative = best
    max_score = max(scores)  # Least negative = worst

    if min_score == max_score:
        return {id: 1.0 for id in bm25_scores}

    score_range = max_score - min_score
    return {
        id: (max_score - score) / score_range
        for id, score in bm25_scores.items()
    }

def normalize_distances(distances: dict[int, float]) -> dict[int, float]:
    """
    Normalize cosine distances to similarity scores in [0, 1].

    Cosine distance is in [0, 2] where 0 = identical.
    We convert to similarity: 1 - (distance / 2)
    Then normalize so best match gets 1.0.
    """
    if not distances:
        return {}

    # Convert distances to similarities
    similarities = {id: 1 - (dist / 2) for id, dist in distances.items()}

    # Normalize to [0, 1] range
    min_sim = min(similarities.values())
    max_sim = max(similarities.values())

    if min_sim == max_sim:
        return {id: 1.0 for id in similarities}

    sim_range = max_sim - min_sim
    return {
        id: (sim - min_sim) / sim_range
        for id, sim in similarities.items()
    }

In [13]:
def get_metadata_by_ids(conn, ids: list[int]) -> dict[int, dict]:
    """Retrieve metadata for given IDs from embeddings_meta table."""
    if not ids:
        return {}

    cursor = conn.cursor()
    placeholders = ",".join("?" * len(ids))
    cursor.execute(f"""
        SELECT id, source_type, source_id, content, metadata
        FROM embeddings_meta
        WHERE id IN ({placeholders})
    """, ids)

    results = {}
    for row in cursor.fetchall():
        results[row[0]] = {
            "source_type": row[1],
            "source_id": row[2],
            "content": row[3],
            "metadata": json.loads(row[4]) if row[4] else {},
        }
    return results

def hybrid_search(
    conn,
    query: str,
    query_embedding: list[float],
    keyword_weight: float = 0.5,
    semantic_weight: float = 0.5,
    top_k: int = 10,
) -> list[dict]:
    """
    Perform hybrid search combining BM25 and sqlite-vec cosine similarity.

    Formula: final_score = keyword_weight * bm25 + semantic_weight * cosine_sim

    Args:
        conn: Database connection
        query: Search query text
        query_embedding: Pre-computed embedding of the query
        keyword_weight: Weight for BM25 (0-1)
        semantic_weight: Weight for cosine similarity (0-1)
        top_k: Number of results to return

    Returns:
        List of results sorted by combined score (highest first)
    """
    # Step 1: Get BM25 scores from FTS5
    bm25_raw = bm25_search(conn, query)
    bm25_normalized = normalize_bm25_scores(bm25_raw)

    # Step 2: Get semantic distances from sqlite-vec
    semantic_raw = semantic_search(conn, query_embedding, limit=100)
    semantic_normalized = normalize_distances(semantic_raw)

    # Step 3: Get all unique IDs from both searches
    all_ids = set(bm25_normalized.keys()) | set(semantic_normalized.keys())

    if not all_ids:
        return []

    # Step 4: Get metadata for all candidates
    metadata = get_metadata_by_ids(conn, list(all_ids))

    # Step 5: Compute combined scores
    scored_results = []

    for id in all_ids:
        # BM25 score (0 if no keyword match)
        bm25_score = bm25_normalized.get(id, 0.0)

        # Semantic score (0 if not in top semantic results)
        semantic_score = semantic_normalized.get(id, 0.0)

        # Combined score
        final_score = (keyword_weight * bm25_score) + (semantic_weight * semantic_score)

        meta = metadata.get(id, {})
        scored_results.append({
            "id": id,
            "content": meta.get("content", ""),
            "source_type": meta.get("source_type", ""),
            "source_id": meta.get("source_id", ""),
            "metadata": meta.get("metadata", {}),
            "bm25_score": bm25_score,
            "semantic_score": semantic_score,
            "final_score": final_score,
        })

    # Sort by final score (descending)
    scored_results.sort(key=lambda x: x["final_score"], reverse=True)

    return scored_results[:top_k]

# Test hybrid search
query_emb = generate_embedding(test_query)

results = hybrid_search(db, test_query, query_emb, top_k=5)

print(f"Hybrid search for '{test_query}':\n")
for i, r in enumerate(results, 1):
    print(f"{i}. Score: {r['final_score']:.3f} (BM25: {r['bm25_score']:.3f}, Semantic: {r['semantic_score']:.3f})")
    print(f"   Source: {r['source_id']}")
    print(f"   Preview: {r['content'][:100]}...\n")

Hybrid search for 'REPLACE WITH A QUERY YOU WANT TO TEST':

1. Score: 0.500 (BM25: 0.000, Semantic: 1.000)
   Source: company.md
   Preview: [From: company.md]
# Our Company

## Team

Our team has 20 years of combined experience....

2. Score: 0.363 (BM25: 0.000, Semantic: 0.725)
   Source: company.md
   Preview: [From: company.md]
# Our Company

# Our Company

We are an AI consulting firm....

3. Score: 0.000 (BM25: 0.000, Semantic: 0.000)
   Source: company.md
   Preview: [From: company.md]
# Our Company

## Services

We offer enterprise AI strategy consulting....



---
## Part 5: Post Generation with RAG Context

Now we combine everything: retrieve relevant context and generate a post.

In [14]:
def format_context_for_prompt(results: list[dict], max_chars: int = 4000) -> str:
    """Format search results into context for the LLM prompt."""
    if not results:
        return "No relevant context found."

    context_parts = []
    chars_used = 0

    for i, result in enumerate(results, 1):
        header = f"[{i}. {result['source_type']}] (score: {result['final_score']:.2f})"
        content = result["content"]

        available = max_chars - chars_used - len(header) - 10
        if available <= 100:
            break

        if len(content) > available:
            content = content[:available - 3] + "..."

        entry = f"{header}\n{content}\n"
        context_parts.append(entry)
        chars_used += len(entry)

    return "\n".join(context_parts)

def retrieve_context(conn, query: str, top_k: int = 10) -> tuple[str, list[dict]]:
    """High-level function to retrieve and format context for RAG."""
    query_embedding = generate_embedding(query)
    results = hybrid_search(conn, query, query_embedding, top_k=top_k)
    formatted = format_context_for_prompt(results)
    return formatted, results

# Test context retrieval
context, results = retrieve_context(db, "AI consulting")
print("Retrieved context:\n")
print(context[:1500] + "..." if len(context) > 1500 else context)

Retrieved context:

[1. business_doc] (score: 1.00)
[From: company.md]
# Our Company

## Services

We offer enterprise AI strategy consulting.

[2. business_doc] (score: 0.39)
[From: company.md]
# Our Company

# Our Company

We are an AI consulting firm.

[3. business_doc] (score: 0.00)
[From: company.md]
# Our Company

## Team

Our team has 20 years of combined experience.



In [ ]:
from google.colab import userdata
import os

os.environ["OPENROUTER_API_KEY"]  = userdata.get("OPENROUTER_API_KEY") or "YOUR_API_KEY_HERE"

In [16]:
import os
from openai import OpenAI


if os.environ.get("OPENROUTER_API_KEY"):
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
    )
    def generate_post_with_rag(context: str, topic: str) -> str:
        """Generate a social media post using RAG context."""
        try:
            response = client.chat.completions.create(
                model="nvidia/nemotron-3-nano-30b-a3b:free",
                max_tokens=1000, # thinking model outputs some thinking tokens first
                messages=[
                    {
                        "role": "system",# TODO: CHANGE SYSTEM PROMPT TO MATCH YOUR COMPANY AND STYLE
                        "content": "You are a social media manager for Truststacksocial, an AI-powered risk operations platform for e-commerce retailers, direct-to-consumer brands. Create engaging Mastodon posts that are concise (under 400 characters), highlight practical AI insights, include 1-2 relevant hashtags, and sound authentic. Use ONLY information from the provided context.",
                    },
                    {
                        "role": "user",
                        "content": f"Based on this context:\n\n{context}\n\nCreate a post about: {topic}\n\nOutput only the post text.",
                    },
                ],
            )
            return response.choices[0].message.content.strip() or ""
        except Exception as e:
            print(f"Error generating post: {e}")
            return ""

    # Generate a post
    topic = "AI-powered risk operation platform" #TODO CHANGE TOPIC TO MATCH YOUR COMPANY
    context, _ = retrieve_context(db, topic)
    post = generate_post_with_rag(context, topic)
    print(f"Generated post ({len(post)} chars):\n{post}")
else:
    print("No API key provided. Skipping post generation.")

No API key provided. Skipping post generation.


---
## Part 6: Full Workflow Demo

Putting it all together: detect changes → retrieve context → generate post

---
## RAG Exercises

Try these exercises to deepen your understanding:

### Exercise 1: Weight Tuning
Experiment with different BM25/semantic weight ratios. When does pure BM25 work better? When does pure semantic work better?

### Exercise 2: Chunking Strategies
Modify `chunk_document()` to chunk by:
- Fixed character count (e.g., 500 chars)
- Paragraph boundaries
- Sentence count


### Exercise 3: Query Expansion
Before searching, use an LLM to expand the query with synonyms. Does this improve results?

### Exercise 4: Reranking
After hybrid search, use an LLM to rerank the top 10 results. Compare to the original ranking.

---
## Project Applications


### 1: Replace local knowledgebase with Notion api to get docs


### 2: Modify Chunking on your docs
Modify `chunk_document()` to chunk by:
- Fixed character count (e.g., 500 chars)
- Paragraph boundaries
- Sentence count

### 4: Add RAG retrieval to create posts function context


### 5: Auto-create posts using Notion API listener

### 6: Auto-reply to comments using Mastodon comments listener

### 7: OPTIONAL: Add posts to sqllite db storage for retrieval